In [1]:
#Setup
from pathlib import Path
from dotenv import load_dotenv
import os
import sys
import json
import time

PROJECT_ROOT = Path(
    "/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Project:", PROJECT_ROOT)
print("Nebius configured:", bool(os.getenv("NEBIUS_API_KEY")))

Project: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Nebius configured: True


In [31]:
#Load both saved results from graph and vector retrieval experiments
graph_results_path = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_retrieval_results.json"
)

vector_results_path = (
    PROJECT_ROOT
    / "evaluation"
    / "vector_retrieval_results.json"
)

graph_retrieval_results = json.loads(
    graph_results_path.read_text()
)

vector_retrieval_results = json.loads(
    vector_results_path.read_text()
)

print("Graph questions:", len(graph_retrieval_results))
print("Vector questions:", len(vector_retrieval_results))

Graph questions: 10
Vector questions: 10


In [3]:
#Configure the shared answer-generation LLM
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1"
)

ANSWER_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"

print("Answer model:", ANSWER_MODEL)

Answer model: Qwen/Qwen3-30B-A3B-Instruct-2507


In [4]:
#Create shared prompt template for answer generation
ANSWER_SYSTEM_PROMPT = """
You are answering questions about the fictional Acme AI organization.

Answer using ONLY the provided retrieved evidence.

Rules:
1. Do not use outside knowledge.
2. Do not invent missing facts.
3. If the evidence is insufficient to answer the question, say:
   "I could not determine this from the retrieved evidence."
4. Be concise and directly answer the question.
5. When multiple people, projects, technologies, or decisions are required,
   include all that are supported by the evidence.
"""

In [5]:
#Format GraphRAG Evidence
def format_graph_evidence(retrieval):
    if not retrieval:
        return "No graph evidence was retrieved."

    lines = []

    for i, row in enumerate(retrieval, start=1):
        parts = [
            f"{key}: {value}"
            for key, value in row.items()
            if value is not None
        ]

        lines.append(
            f"Graph result {i}: "
            + " | ".join(parts)
        )

    return "\n".join(lines)

In [6]:
#Test
example = graph_retrieval_results[3]

print(example["question"])
print()
print(
    format_graph_evidence(
        example["retrieval"]
    )
)

Who worked on Project Phoenix and also has Kubernetes experience?

Graph result 1: person: Bob Singh | project: Project Phoenix | skill: Kubernetes
Graph result 2: person: Hannah Brooks | project: Project Phoenix | skill: Kubernetes


In [7]:
#Formart VectorRAG Evidence
def format_vector_evidence(retrieval):
    if not retrieval:
        return "No vector evidence was retrieved."

    blocks = []

    for item in retrieval:

        blocks.append(
            f"""
Source {item['rank']}: {item['file_name']}
Similarity score: {item['score']:.4f}

{item['text']}
""".strip()
        )

    return "\n\n---\n\n".join(blocks)

In [8]:
#Test
example = vector_retrieval_results[3]

print(example["question"])
print()
print(
    format_vector_evidence(
        example["retrieval"]
    )[:3000]
)

Who worked on Project Phoenix and also has Kubernetes experience?

Source 1: project_phoenix.md
Similarity score: 0.8952

---
doc_id: doc_project_phoenix
doc_type: project_overview
entity_id: project_phoenix
owner: Recommendations
---
# Project Phoenix

**Status:** Production  
**Owning team:** Recommendations  
**Project lead:** Farah Khan  
**Core technologies:** Kafka, Kubernetes, Redis, Python

## Purpose
Serve real-time personalized product recommendations using streaming behavior and low-latency feature access.

## Contributors
Alice Chen, Bob Singh, Carol Martinez, Farah Khan, Hannah Brooks, Julia Patel

---

Source 2: kubernetes_service_standard.md
Similarity score: 0.8920

---
doc_id: doc_tech_kubernetes_standard
doc_type: technical_document
owner: Data Platform
---
# Kubernetes Service Deployment Standard

Kubernetes is the preferred deployment platform for continuously running services that need controlled rollouts, autoscaling, and standardized health checks.

Hannah Brooks

In [9]:
#Shared answer-generation function
def generate_answer(
    question: str,
    evidence: str,
):
    user_prompt = f"""
Question:
{question}

Retrieved evidence:
{evidence}

Answer:
"""

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[
            {
                "role": "system",
                "content": ANSWER_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0.0,
        max_tokens=250,
    )
#capture latency too because the project handout specifically calls latency a first-class RAG consideration
    latency = time.perf_counter() - start

    answer = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    usage = response.usage

    return {
        "answer": answer,
        "latency_seconds": latency,
        "input_tokens": (
            usage.prompt_tokens
            if usage else None
        ),
        "output_tokens": (
            usage.completion_tokens
            if usage else None
        ),
    }

In [10]:
#Test one GraphRAG answer first
item = graph_retrieval_results[3]

graph_evidence = format_graph_evidence(
    item["retrieval"]
)

test_answer = generate_answer(
    question=item["question"],
    evidence=graph_evidence,
)

print("Question:")
print(item["question"])

print("\nExpected:")
print(item["expected_answer"])

print("\nGraphRAG:")
print(test_answer["answer"])

print(
    "\nLatency:",
    round(
        test_answer["latency_seconds"],
        2
    ),
    "seconds"
)

print(
    "Tokens:",
    test_answer["input_tokens"],
    "→",
    test_answer["output_tokens"]
)

Question:
Who worked on Project Phoenix and also has Kubernetes experience?

Expected:
['Bob Singh', 'Hannah Brooks']

GraphRAG:
Bob Singh and Hannah Brooks worked on Project Phoenix and have Kubernetes experience.

Latency: 1.34 seconds
Tokens: 169 → 15


In [11]:
#Test same question with VectorRAG answer
item = vector_retrieval_results[3]

vector_evidence = format_vector_evidence(
    item["retrieval"]
)

test_vector_answer = generate_answer(
    question=item["question"],
    evidence=vector_evidence,
)

print("Question:")
print(item["question"])

print("\nExpected:")
print(item["expected_answer"])

print("\nVectorRAG:")
print(test_vector_answer["answer"])

print(
    "\nLatency:",
    round(
        test_vector_answer["latency_seconds"],
        2
    ),
    "seconds"
)

Question:
Who worked on Project Phoenix and also has Kubernetes experience?

Expected:
['Bob Singh', 'Hannah Brooks']

VectorRAG:
Bob Singh and Hannah Brooks worked on Project Phoenix and have Kubernetes experience.

Latency: 0.97 seconds


In [33]:
#Generate GraphRAG answers for all questions
graph_answers = []

for item in graph_retrieval_results:

    print(
        f"Generating GraphRAG Q{item['id']}..."
    )

    evidence = format_graph_evidence(
        item["retrieval"]
    )

    generation = generate_answer(
        question=item["question"],
        evidence=evidence,
    )

    graph_answers.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": item.get("route"),
        "answer": generation["answer"],
        "latency_seconds":
            generation["latency_seconds"],
        "input_tokens":
            generation["input_tokens"],
        "output_tokens":
            generation["output_tokens"],
    })

print("✅ GraphRAG answers:", len(graph_answers))

Generating GraphRAG Q1...
Generating GraphRAG Q2...
Generating GraphRAG Q3...
Generating GraphRAG Q4...
Generating GraphRAG Q5...
Generating GraphRAG Q6...
Generating GraphRAG Q7...
Generating GraphRAG Q8...
Generating GraphRAG Q9...
Generating GraphRAG Q10...
✅ GraphRAG answers: 10


In [34]:
#Generate VectorRAG answers for all questions
vector_answers = []

for item in vector_retrieval_results:

    print(
        f"Generating VectorRAG Q{item['id']}..."
    )

    evidence = format_vector_evidence(
        item["retrieval"]
    )

    generation = generate_answer(
        question=item["question"],
        evidence=evidence,
    )

    vector_answers.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "answer": generation["answer"],
        "latency_seconds":
            generation["latency_seconds"],
        "input_tokens":
            generation["input_tokens"],
        "output_tokens":
            generation["output_tokens"],
    })

print("✅ VectorRAG answers:", len(vector_answers))


Generating VectorRAG Q1...
Generating VectorRAG Q2...
Generating VectorRAG Q3...
Generating VectorRAG Q4...
Generating VectorRAG Q5...
Generating VectorRAG Q6...
Generating VectorRAG Q7...
Generating VectorRAG Q8...
Generating VectorRAG Q9...
Generating VectorRAG Q10...
✅ VectorRAG answers: 10


In [35]:
#Save both results
graph_answer_path = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_rag_answers.json"
)

vector_answer_path = (
    PROJECT_ROOT
    / "evaluation"
    / "vector_rag_answers.json"
)

graph_answer_path.write_text(
    json.dumps(
        graph_answers,
        indent=2,
    )
)

vector_answer_path.write_text(
    json.dumps(
        vector_answers,
        indent=2,
    )
)

print("Saved:")
print(graph_answer_path)
print(vector_answer_path)

Saved:
/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/graph_rag_answers.json
/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/vector_rag_answers.json


In [36]:
#Side-by-side inspection
for graph_item, vector_item in zip(
    graph_answers,
    vector_answers,
):

    print("=" * 100)

    print(
        f"Q{graph_item['id']} "
        f"[{graph_item['category']}]"
    )

    print("\nQuestion:")
    print(graph_item["question"])

    print("\nExpected:")
    print(graph_item["expected_answer"])

    print("\nGraphRAG:")
    print(graph_item["answer"])

    print("\nVectorRAG:")
    print(vector_item["answer"])

    print()

Q1 [single_fact]

Question:
What database does Project Phoenix use for its online feature cache?

Expected:
Redis

GraphRAG:
Project Phoenix uses Redis for its online feature cache.

VectorRAG:
Project Phoenix uses Redis for its online feature cache.

Q2 [single_fact]

Question:
What is the purpose of Project Atlas?

Expected:
Modernize product search ranking with fresh behavioral signals and a unified retrieval stack.

GraphRAG:
The purpose of Project Atlas is to modernize product search ranking with fresh behavioral signals and a unified retrieval stack.

VectorRAG:
The purpose of Project Atlas is to modernize product search ranking with fresh behavioral signals and a unified retrieval stack.

Q3 [semantic]

Question:
Why did Project Atlas adopt Kafka?

Expected:
Atlas needed replayable, ordered product and behavior updates and wanted to decouple producers from ranking consumers.

GraphRAG:
Project Atlas adopted Kafka to enable replayable, ordered product and behavior updates, levera

In [37]:
#Deterministic evaluation of answers
import re


def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def deterministic_score(
    answer,
    expected_answer,
):
    """
    Returns a simple expected-fact coverage score.

    For list answers:
        fraction of expected items mentioned.

    For string answers:
        1 if expected text appears in answer,
        otherwise 0.

    This is a heuristic, especially for narrative answers.
    """

    answer_norm = normalize_text(answer)

    # Expected answer is a list
    if isinstance(expected_answer, list):

        matches = []

        for item in expected_answer:

            item_norm = normalize_text(item)

            found = item_norm in answer_norm

            matches.append({
                "expected": item,
                "found": found,
            })

        score = (
            sum(x["found"] for x in matches)
            / len(matches)
            if matches
            else 0
        )

        return {
            "score": score,
            "details": matches,
        }

    # Expected answer is a string
    expected_norm = normalize_text(
        expected_answer
    )

    found = expected_norm in answer_norm

    return {
        "score": 1.0 if found else 0.0,
        "details": [
            {
                "expected": expected_answer,
                "found": found,
            }
        ],
    }

In [38]:
#Score GraphRAG and VectorRAG answers
deterministic_results = []

for graph_item, vector_item in zip(
    graph_answers,
    vector_answers,
):

    graph_score = deterministic_score(
        graph_item["answer"],
        graph_item["expected_answer"],
    )

    vector_score = deterministic_score(
        vector_item["answer"],
        vector_item["expected_answer"],
    )

    deterministic_results.append({
        "id": graph_item["id"],
        "category": graph_item["category"],
        "question": graph_item["question"],

        "graph_score":
            graph_score["score"],

        "vector_score":
            vector_score["score"],

        "graph_answer":
            graph_item["answer"],

        "vector_answer":
            vector_item["answer"],

        "expected_answer":
            graph_item["expected_answer"],
    })

In [39]:
for item in deterministic_results:

    print("=" * 90)

    print(
        f"Q{item['id']} "
        f"[{item['category']}]"
    )

    print(
        "Graph:",
        f"{item['graph_score']:.0%}"
    )

    print(
        "Vector:",
        f"{item['vector_score']:.0%}"
    )

Q1 [single_fact]
Graph: 100%
Vector: 100%
Q2 [single_fact]
Graph: 100%
Vector: 100%
Q3 [semantic]
Graph: 0%
Vector: 0%
Q4 [relationship]
Graph: 100%
Vector: 100%
Q5 [relationship]
Graph: 100%
Vector: 100%
Q6 [multi_hop]
Graph: 100%
Vector: 100%
Q7 [relationship]
Graph: 100%
Vector: 50%
Q8 [multi_hop]
Graph: 0%
Vector: 100%
Q9 [multi_hop]
Graph: 100%
Vector: 100%
Q10 [multi_hop]
Graph: 40%
Vector: 40%


In [19]:
#Configure an LLM judge
JUDGE_MODEL = "openai/gpt-oss-120b"

print("Judge:", JUDGE_MODEL)

Judge: openai/gpt-oss-120b


We're going to make the judge evaluate:

Correctness against known ground truth
Completeness
Faithfulness to retrieved evidence

That last one is particularly important for RAG.

In [46]:
#Create the judge function
import json


def judge_answer(
    question,
    expected_answer,
    evidence,
    answer,
):
    prompt = f"""
You are evaluating a RAG system.

QUESTION:
{question}

GROUND TRUTH:
{json.dumps(expected_answer)}

RETRIEVED EVIDENCE:
{evidence}

GENERATED ANSWER:
{answer}

Evaluate the generated answer.

Score each dimension from 0 to 2:

Correctness:
0 = incorrect
1 = partially correct
2 = fully correct

Completeness:
0 = important information missing
1 = partially complete
2 = fully complete

Faithfulness:
0 = claims are unsupported by evidence
1 = partially supported
2 = fully supported

Return ONLY a JSON object with exactly these keys:

{{
  "correctness": 0,
  "completeness": 0,
  "faithfulness": 0,
  "reason": "short explanation"
}}
"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,

        # More room than the earlier 250
        max_tokens=1500,

        # Force JSON output
        response_format={
            "type": "json_object"
        },
    )

    text = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    try:
        return json.loads(text)

    except json.JSONDecodeError as e:
        print("❌ Invalid JSON returned by judge")
        print("Error:", e)

        print("\nRAW MODEL OUTPUT:")
        print(text)

        raise

In [47]:
#Test the judge on one question
#Q4
q_index = 3 #For Q4

graph_item = graph_answers[q_index]
graph_retrieval = graph_retrieval_results[q_index]

graph_evidence = format_graph_evidence(
    graph_retrieval["retrieval"]
)

graph_judgment = judge_answer(
    question=graph_item["question"],
    expected_answer=
        graph_item["expected_answer"],
    evidence=graph_evidence,
    answer=graph_item["answer"],
)

graph_judgment

{'correctness': 2,
 'completeness': 2,
 'faithfulness': 2,
 'reason': 'The answer lists both Bob Singh and Hannah Brooks, matching the ground truth, includes all required individuals, and each claim is directly supported by the retrieved evidence.'}

In [48]:
#Judge all questions for both GraphRAG and VectorRAG
judge_results = []

for i in range(len(graph_answers)):

    graph_item = graph_answers[i]
    vector_item = vector_answers[i]

    graph_retrieval = graph_retrieval_results[i]
    vector_retrieval = vector_retrieval_results[i]

    print(f"\nJudging Q{graph_item['id']}...")

    # -----------------------------
    # GraphRAG
    # -----------------------------

    graph_evidence = format_graph_evidence(
        graph_retrieval["retrieval"]
    )

    try:
        graph_judge = judge_answer(
            question=graph_item["question"],
            expected_answer=graph_item["expected_answer"],
            evidence=graph_evidence,
            answer=graph_item["answer"],
        )

    except Exception as e:
        print(
            f"❌ Graph judge failed for Q{graph_item['id']}:",
            e
        )

        graph_judge = None

    # -----------------------------
    # VectorRAG
    # -----------------------------

    vector_evidence = format_vector_evidence(
        vector_retrieval["retrieval"]
    )

    try:
        vector_judge = judge_answer(
            question=vector_item["question"],
            expected_answer=vector_item["expected_answer"],
            evidence=vector_evidence,
            answer=vector_item["answer"],
        )

    except Exception as e:
        print(
            f"❌ Vector judge failed for Q{vector_item['id']}:",
            e
        )

        vector_judge = None

    judge_results.append({
        "id": graph_item["id"],
        "category": graph_item["category"],
        "question": graph_item["question"],

        "graph": graph_judge,
        "vector": vector_judge,

        "graph_latency":
            graph_item["latency_seconds"],

        "vector_latency":
            vector_item["latency_seconds"],
    })

print(
    "\n✅ Questions processed:",
    len(judge_results)
)


Judging Q1...

Judging Q2...

Judging Q3...

Judging Q4...

Judging Q5...

Judging Q6...

Judging Q7...

Judging Q8...

Judging Q9...

Judging Q10...

✅ Questions processed: 10


In [51]:
#Check for failures
for item in judge_results:

    if (
        item["graph"] is None
        or item["vector"] is None
    ):
        print(
            "Needs retry:",
            f"Q{item['id']}"
        )
    else:
            print(
            "Judged:",
            f"Q{item['id']}"
            )

Judged: Q1
Judged: Q2
Judged: Q3
Judged: Q4
Judged: Q5
Judged: Q6
Judged: Q7
Judged: Q8
Judged: Q9
Judged: Q10


In [45]:
#Test why it failed
graph_item = graph_answers[9]
graph_retrieval = graph_retrieval_results[9]

graph_evidence = format_graph_evidence(
    graph_retrieval["retrieval"]
)

prompt = f"""
You are evaluating a RAG system.

QUESTION:
{graph_item["question"]}

GROUND TRUTH:
{json.dumps(graph_item["expected_answer"])}

RETRIEVED EVIDENCE:
{graph_evidence}

GENERATED ANSWER:
{graph_item["answer"]}

Score each from 0 to 2:

Correctness
Completeness
Faithfulness

Return ONLY valid JSON:

{{
  "correctness": 0,
  "completeness": 0,
  "faithfulness": 0,
  "reason": "brief explanation"
}}
"""

response = client.chat.completions.create(
    model=JUDGE_MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    temperature=0.0,
    max_tokens=1000,
    response_format={
        "type": "json_object"
    },
)

print("Finish reason:", response.choices[0].finish_reason)
print("Message:", response.choices[0].message)
print("Usage:", response.usage)

Finish reason: stop
Message: ChatCompletionMessage(content='{\n  "correctness": 1,\n  "completeness": 0,\n  "faithfulness": 0,\n  "reason": "The answer correctly lists the two Phoenix decisions but omits the three Atlas decisions that are part of the ground truth, making it incomplete. It also incorrectly claims no other overlapping team members are present, which is not faithful to the expected answer."\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='We need to evaluate the generated answer against ground truth. Ground truth list: ["Adopt Kafka for Atlas change feed", "Standardize Elasticsearch mappings for Atlas", "Deploy Atlas ranking services on Kubernetes", "Use Redis for Phoenix online feature cache", "Use Kafka as Phoenix behavior event bus"]\n\nThe question: "Which architecture decisions affected projects owned by teams whose members also worked on Project Phoenix?" So we need decisions that affect projects owned 

In [52]:
#Reload corrected evaluation questions

import json

questions_path = (
    PROJECT_ROOT
    / "evaluation"
    / "questions.json"
)

evaluation_questions = json.loads(
    questions_path.read_text()
)

q10 = next(
    q for q in evaluation_questions
    if q["id"] == 10
)

print("Updated Q10 expected answer:")
for item in q10["expected_answer"]:
    print("-", item)

Updated Q10 expected answer:
- Adopt Kafka for Atlas change feed
- Standardize Elasticsearch mappings for Atlas
- Deploy Atlas ranking services on Kubernetes
- Use Redis for Phoenix online feature cache
- Use Kafka as Phoenix behavior event bus
- Use Airflow for Mercury settlement workflows
- Use PostgreSQL as Mercury reconciliation ledger


In [53]:
#Calculate normalized scores
for item in judge_results:

    graph = item["graph"]
    vector = item["vector"]

    item["graph_score"] = (
        graph["correctness"]
        + graph["completeness"]
        + graph["faithfulness"]
    ) / 6

    item["vector_score"] = (
        vector["correctness"]
        + vector["completeness"]
        + vector["faithfulness"]
    ) / 6

In [54]:
#Determine the winner
for item in judge_results:

    difference = (
        item["graph_score"]
        - item["vector_score"]
    )

    if difference > 0.05:
        winner = "GraphRAG"

    elif difference < -0.05:
        winner = "VectorRAG"

    else:
        winner = "Tie"

    item["winner"] = winner

In [55]:
#Display the comparison
import pandas as pd

comparison_df = pd.DataFrame([
    {
        "Q": item["id"],
        "Category": item["category"],

        "Graph Score":
            round(
                item["graph_score"] * 100,
                1
            ),

        "Vector Score":
            round(
                item["vector_score"] * 100,
                1
            ),

        "Winner":
            item["winner"],

        "Graph Latency":
            round(
                item["graph_latency"],
                2
            ),

        "Vector Latency":
            round(
                item["vector_latency"],
                2
            ),
    }

    for item in judge_results
])

comparison_df

,Q,Category,Graph Score,Vector Score,Winner,Graph Latency,Vector Latency
0,1,single_fact,100.0,100.0,Tie,3.49,3.77
1,2,single_fact,100.0,100.0,Tie,0.70,1.35
2,3,semantic,100.0,100.0,Tie,0.95,0.97
3,4,relationship,100.0,100.0,Tie,1.46,0.63
4,5,relationship,83.3,100.0,VectorRAG,0.79,1.30
5,6,multi_hop,100.0,100.0,Tie,0.79,0.74
6,7,relationship,100.0,33.3,GraphRAG,2.21,5.77
7,8,multi_hop,33.3,100.0,VectorRAG,0.44,2.22
8,9,multi_hop,100.0,100.0,Tie,1.65,1.93
9,10,multi_hop,66.7,16.7,GraphRAG,2.24,4.17


In [56]:
#Summary by query category
category_summary = (
    comparison_df
    .groupby("Category")
    .agg({
        "Graph Score": "mean",
        "Vector Score": "mean",
        "Graph Latency": "mean",
        "Vector Latency": "mean",
    })
    .round(1)
)

category_summary

,Graph Score,Vector Score,Graph Latency,Vector Latency
Category,,,,
multi_hop,75.0,79.2,1.3,2.3
relationship,94.4,77.8,1.5,2.6
semantic,100.0,100.0,1.0,1.0
single_fact,100.0,100.0,2.1,2.6


In [57]:
#Count winners
winner_counts = (
    comparison_df["Winner"]
    .value_counts()
)

winner_counts

Winner
Tie          6
VectorRAG    2
GraphRAG     2
Name: count, dtype: int64

In [58]:
#Save results to JSON
judge_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "rag_comparison_results.json"
)

judge_output_path.write_text(
    json.dumps(
        judge_results,
        indent=2,
    )
)

csv_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "rag_comparison_summary.csv"
)

comparison_df.to_csv(
    csv_output_path,
    index=False,
)

print("Saved:")
print(judge_output_path)
print(csv_output_path)

Saved:
/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/rag_comparison_results.json
/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/rag_comparison_summary.csv


In [59]:
#Detailed per-question comparison
for item in judge_results:

    print("=" * 100)

    print(
        f"Q{item['id']} "
        f"[{item['category']}]"
    )

    print(item["question"])

    print(
        f"\nGraph Score:  "
        f"{item['graph_score'] * 100:.1f}%"
    )

    print(
        f"Vector Score: "
        f"{item['vector_score'] * 100:.1f}%"
    )

    print(
        "\nWinner:",
        item["winner"]
    )

    print(
        "\nGraph judge reason:"
    )
    print(
        item["graph"]["reason"]
    )

    print(
        "\nVector judge reason:"
    )
    print(
        item["vector"]["reason"]
    )

Q1 [single_fact]
What database does Project Phoenix use for its online feature cache?

Graph Score:  100.0%
Vector Score: 100.0%

Winner: Tie

Graph judge reason:
The answer correctly states that Project Phoenix uses Redis, fully addresses the question, and is directly supported by the retrieved evidence.

Vector judge reason:
The answer correctly states Redis, which fully answers the question, includes all required information, and is directly supported by multiple retrieved documents.
Q2 [single_fact]
What is the purpose of Project Atlas?

Graph Score:  100.0%
Vector Score: 100.0%

Winner: Tie

Graph judge reason:
The answer exactly matches the ground truth purpose and is directly supported by the retrieved evidence.

Vector judge reason:
The answer exactly matches the ground truth purpose statement and is directly supported by the retrieved project overview and technical documents.
Q3 [semantic]
Why did Project Atlas adopt Kafka?

Graph Score:  100.0%
Vector Score: 100.0%

Winner: T

In [60]:
#Performance by question type
category_summary = (
    comparison_df
    .groupby("Category")
    .agg(
        Graph_Score=("Graph Score", "mean"),
        Vector_Score=("Vector Score", "mean"),
        Graph_Latency=("Graph Latency", "mean"),
        Vector_Latency=("Vector Latency", "mean"),
    )
    .round(1)
)

category_summary

,Graph_Score,Vector_Score,Graph_Latency,Vector_Latency
Category,,,,
multi_hop,75.0,79.2,1.3,2.3
relationship,94.4,77.8,1.5,2.6
semantic,100.0,100.0,1.0,1.0
single_fact,100.0,100.0,2.1,2.6


In [61]:
#Overall averages
print(
    "Average GraphRAG score:",
    round(
        comparison_df["Graph Score"].mean(),
        1
    )
)

print(
    "Average VectorRAG score:",
    round(
        comparison_df["Vector Score"].mean(),
        1
    )
)

print(
    "Average GraphRAG latency:",
    round(
        comparison_df["Graph Latency"].mean(),
        2
    )
)

print(
    "Average VectorRAG latency:",
    round(
        comparison_df["Vector Latency"].mean(),
        2
    )
)

Average GraphRAG score: 88.3
Average VectorRAG score: 85.0
Average GraphRAG latency: 1.47
Average VectorRAG latency: 2.28
